# Offline latent pipeline ledger

The notebook checks latent shapes, guidance, and LoRA matrices; it never downloads a model.

In [ ]:
from pathlib import Path
import importlib.util
lesson_rel = Path('phases/04-computer-vision/11-stable-diffusion')
candidates = []
for start in [Path.cwd(), *Path.cwd().parents]:
    candidates.append(start / lesson_rel / 'code/main.py')
code_path = next(path.resolve() for path in candidates if path.is_file())
spec = importlib.util.spec_from_file_location('cv04_l11', code_path)
sd = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sd)
print(code_path)

In [ ]:
import numpy as np
image = np.zeros((1, 3, 32, 32))
latent = sd.encode_latent(image, downsample_factor=8, latent_channels=4)
guided = sd.classifier_free_guidance(np.zeros_like(latent), np.ones_like(latent), 5)
print('latent', latent.shape, 'guided mean', guided.mean())

In [ ]:
updated = sd.lora_update(np.zeros((4, 4)), np.ones((2, 4)), np.ones((4, 2)), scale=0.5)
sigmas = sd.scheduler_sigmas(2, 1.0, 0.1)
assert np.allclose(sigmas, [1.0, 0.1])
print('lora update', updated.shape, 'sigmas', sigmas.tolist(), 'components', len(sd.pipeline_manifest()))